In [6]:
import pandas as pd
import yaml
import os
import openpyxl
import pygwalker as pyg
import matplotlib.pyplot as plt
import seaborn as sns
import re
import sys
from pathlib import Path
from datetime import datetime

In [8]:
# To add project files
# Keeps going up project structure until it gets to the root.
# Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69a5ff07-91e0-8329-88d1-cf2ea6c159c2
project_root = Path().resolve()
while project_root.name != "AFOLU_GHG_flux_model":
    project_root = project_root.parent

sys.path.append(str(project_root))

from src.utilities import constants_and_names as cn
from src.utilities import universal_utilities as uu

now = datetime.now()
today = now.strftime("%Y%m%d")

pd.set_option('display.max_columns', None)

In [6]:
# SOC (including mineral soil) zonal stats output
zonal_stats_folder = f'/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/SOC_v{cn.SOC_model_version_underscore}_standard_global/'
parquet_name = f'SOC_zonal_stats_v{cn.SOC_model_version_underscore}_20260530_20_53_22.parquet'

In [13]:
%%time

# Reads gross outputs parquet table
df = pd.read_parquet(f'{zonal_stats_folder}{parquet_name}')
print(f"Rows in df: {len(df)}")  # Should equal 7,987,511 per zonal stats log
df

Rows in df: 7987511
CPU times: user 5.34 s, sys: 1.17 s, total: 6.52 s
Wall time: 1.94 s


,analysis_layer,adm0,WDPA,cont_eco,Landmark,starting_composite_primary_forest,KBA,watershed,drivers_of_TCL_1_km,year,value,tile_id,area_ha,gas,country_name,region_L1,region_L2_L3,continent,ecozone,continent_ecozone,climate_domain,watershed_name,WDPA_type,WDPA_high_protection,driver_1km_text,density__Mg_ha
0,SOC_density__full_extent__0-30cm_MgC_ha,NA,0,1020,0,0,0,0,0,2005,232.500580,00N_000E,429584.187500,CO2,no_country,Unassigned,Unassigned,Africa,Tropical rainforest,Africa-Tropical rainforest,Subtropical/tropical,Unassigned,NA,Not protected,Unassigned,0.000541
1,SOC_density__full_extent__0-30cm_MgC_ha,NA,0,1020,0,0,0,0,0,2010,238.314621,00N_000E,429584.187500,CO2,no_country,Unassigned,Unassigned,Africa,Tropical rainforest,Africa-Tropical rainforest,Subtropical/tropical,Unassigned,NA,Not protected,Unassigned,0.000555
2,SOC_density__full_extent__0-30cm_MgC_ha,NA,0,1020,0,0,0,0,0,2015,271.096375,00N_000E,429584.187500,CO2,no_country,Unassigned,Unassigned,Africa,Tropical rainforest,Africa-Tropical rainforest,Subtropical/tropical,Unassigned,NA,Not protected,Unassigned,0.000631
3,SOC_density__full_extent__0-30cm_MgC_ha,NA,0,1020,0,0,0,0,0,2020,365.058746,00N_000E,429584.187500,CO2,no_country,Unassigned,Unassigned,Africa,Tropical rainforest,Africa-Tropical rainforest,Subtropical/tropical,Unassigned,NA,Not protected,Unassigned,0.000850
4,SOC_density__full_extent__0-30cm_MgC_ha,NA,0,1020,0,0,0,0,0,2022,460.359619,00N_000E,429584.187500,CO2,no_country,Unassigned,Unassigned,Africa,Tropical rainforest,Africa-Tropical rainforest,Subtropical/tropical,Unassigned,NA,Not protected,Unassigned,0.001072
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7987506,SOC_gain__mineral_soil_extent__0-30cm_MgCO2,USA,0,2005,1,0,1,1010,0,2015,-109.774841,80N_170W,17205.082031,CO2,USA,North America,Northern America,America North,Polar,America North-Polar,Boreal,Pacific and Arctic Coast,NA,Not protected,Unassigned,-0.006380
7987507,SOC_gain__mineral_soil_extent__0-30cm_MgCO2,USA,0,2005,1,0,1,1010,0,2020,-94.486801,80N_170W,17205.082031,CO2,USA,North America,Northern America,America North,Polar,America North-Polar,Boreal,Pacific and Arctic Coast,NA,Not protected,Unassigned,-0.005492
7987508,SOC_gain__mineral_soil_extent__0-30cm_MgCO2,USA,0,2005,1,0,1,1010,0,2022,-193.210693,80N_170W,17205.082031,CO2,USA,North America,Northern America,America North,Polar,America North-Polar,Boreal,Pacific and Arctic Coast,NA,Not protected,Unassigned,-0.011230
7987509,SOC_gain__mineral_soil_extent__0-30cm_MgCO2,USA,0,2005,1,0,1,1010,7,2010,-0.121576,80N_170W,4.289277,CO2,USA,North America,Northern America,America North,Polar,America North-Polar,Boreal,Pacific and Arctic Coast,NA,Not protected,Other natural disturbances,-0.028344


In [42]:
# QC for df-- to check that all columns have expected options for values
# Not worried about "IOPub data rate exceeded." warnings.

print(f"Columns in df are: {df.columns}\n")
for column in df.columns:
    col_vals = (
        df[column]
        .dropna()
        .sort_values()
        .unique()
        .tolist()
    )
    print(f"{column} ({len(col_vals)} values): {col_vals}\n")

Columns in df are: Index(['analysis_layer', 'adm0', 'WDPA', 'cont_eco', 'Landmark',
       'starting_composite_primary_forest', 'KBA', 'watershed',
       'drivers_of_TCL_1_km', 'year', 'value', 'tile_id', 'area_ha', 'gas',
       'country_name', 'region_L1', 'region_L2_L3', 'continent', 'ecozone',
       'continent_ecozone', 'climate_domain', 'watershed_name', 'WDPA_type',
       'WDPA_high_protection', 'driver_1km_text', 'density__Mg_ha'],
      dtype='object')

analysis_layer (8 options): ['SOC_density__full_extent__0-30cm_MgC_ha', 'SOC_density__mineral_soil_extent__0-30cm_MgC_ha', 'SOC_gain__full_extent__0-30cm_MgCO2', 'SOC_gain__mineral_soil_extent__0-30cm_MgCO2', 'SOC_loss__full_extent__0-30cm_MgCO2', 'SOC_loss__mineral_soil_extent__0-30cm_MgCO2', 'SOC_net__full_extent__0-30cm_MgCO2', 'SOC_net__mineral_soil_extent__0-30cm_MgCO2']

adm0 (217 options): ['ABW', 'AFG', 'AGO', 'AIA', 'ALA', 'ALB', 'AND', 'ARE', 'ARG', 'ARM', 'ATA', 'ATF', 'ATG', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



tile_id (275 options): ['00N_000E', '00N_010E', '00N_020E', '00N_030E', '00N_040E', '00N_040W', '00N_050W', '00N_060W', '00N_070W', '00N_080W', '00N_090E', '00N_090W', '00N_100E', '00N_100W', '00N_110E', '00N_120E', '00N_130E', '00N_140E', '00N_150E', '00N_160E', '10N_000E', '10N_010E', '10N_010W', '10N_020E', '10N_020W', '10N_030E', '10N_040E', '10N_050E', '10N_050W', '10N_060W', '10N_070E', '10N_070W', '10N_080E', '10N_080W', '10N_090E', '10N_090W', '10N_100E', '10N_100W', '10N_110E', '10N_120E', '10N_130E', '10S_010E', '10S_020E', '10S_030E', '10S_040E', '10S_040W', '10S_050E', '10S_050W', '10S_060W', '10S_070W', '10S_080W', '10S_110E', '10S_120E', '10S_130E', '10S_140E', '10S_150E', '10S_160E', '10S_170E', '20N_000E', '20N_010E', '20N_010W', '20N_020E', '20N_020W', '20N_030E', '20N_030W', '20N_040E', '20N_050E', '20N_060W', '20N_070E', '20N_070W', '20N_080E', '20N_080W', '20N_090E', '20N_090W', '20N_100E', '20N_100W', '20N_110E', '20N_110W', '20N_120E', '20S_010E', '20S_020E', '20S

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



gas (1 options): ['CO2']

country_name (217 options): ['Afghanistan', 'Albania', 'Algeria', 'Andorra', 'Angola', 'Anguilla', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Aruba', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bermuda', 'Bhutan', 'Bolivia', 'Bonaire', 'Bosnia and Herzegovina', 'Botswana', 'Brazil', 'British Virgin Islands', 'Brunei', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cambodia', 'Cameroon', 'Canada', 'Cape Verde', 'Cayman Islands', 'Central African Republic', 'Chad', 'Chile', 'China', 'Colombia', 'Comoros', 'Costa Rica', 'Croatia', 'Cuba', 'Curaçao', 'Cyprus', 'Czechia', 'Côte d Ivoire', 'DR Congo', 'Denmark', 'Djibouti', 'Dominica', 'Dominican Republic', 'East Timor', 'Ecuador', 'Egypt', 'El Salvador', 'Equatorial Guinea', 'Eritrea', 'Estonia', 'Ethiopia', 'Fiji', 'Finland', 'France', 'French Guiana', 'French Southern Territories', 'Gabon', 'Gambia', 'Georgia', 'Germany', 'Ghana', '

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



Create simplified/abbreviated table that can be used in PyGWalker

In [81]:
import rasterio
with rasterio.open("s3://gfw2-data/analyses/area_28m/hanson_2013_area_70N_070W.tif") as src:
    print("pixel area transform:", src.transform)
    print("pixel area shape:", src.shape)

with rasterio.open("s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs_soil_organic_carbon/version_1_0_1__standard__global/SOC_gain__mineral_soil_extent__0-30cm_MgCO2/2015/_pixel_yr/40000_pixels/20260526/70N_070W__SOC_gain__mineral_soil_extent__0-30cm_MgCO2_pixel_yr_2015.tif") as src:
    print("SOC COG transform:", src.transform)
    print("SOC COG shape:", src.shape)

with rasterio.open("s3://gfw2-data/analyses/area_28m/hanson_2013_area_70N_070E.tif") as src:
    print("pixel area transform:", src.transform)
    print("pixel area shape:", src.shape)

with rasterio.open("s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs_soil_organic_carbon/version_1_0_1__standard__global/SOC_gain__mineral_soil_extent__0-30cm_MgCO2/2015/_pixel_yr/40000_pixels/20260526/70N_070E__SOC_gain__mineral_soil_extent__0-30cm_MgCO2_pixel_yr_2015.tif") as src:
    print("SOC COG transform:", src.transform)
    print("SOC COG shape:", src.shape)

pixel area transform: | 0.00, 0.00,-70.00|
| 0.00,-0.00, 70.00|
| 0.00, 0.00, 1.00|
pixel area shape: (40000, 40000)
SOC COG transform: | 0.00, 0.00,-70.00|
| 0.00,-0.00, 70.00|
| 0.00, 0.00, 1.00|
SOC COG shape: (40000, 40000)
pixel area transform: | 0.00, 0.00, 70.00|
| 0.00,-0.00, 70.00|
| 0.00, 0.00, 1.00|
pixel area shape: (40000, 40000)
SOC COG transform: | 0.00, 0.00, 70.00|
| 0.00,-0.00, 70.00|
| 0.00, 0.00, 1.00|
SOC COG shape: (40000, 40000)


In [10]:
import rasterio
import fsspec
import zarr
import numpy as np

# GeoTIF values (first 5 rows, first 5 cols of tile)
with rasterio.open("s3://gfw2-data/analyses/area_28m/hanson_2013_area_70N_070W.tif") as src:
    print("GeoTIF transform repr:", repr(src.transform))
    geotif_vals = src.read(1, window=rasterio.windows.Window(0, 0, 5, 5))
    print("GeoTIF top-left 5x5 values:", geotif_vals)

# Zarr values at same location
fs = fsspec.filesystem("s3", anon=False)
store = zarr.open_group(fs.get_mapper(cn.pixel_area_zarr_path), mode="r")
lat_arr = store["y"][:]
lon_arr = store["x"][:]
y0 = np.searchsorted(lat_arr[::-1], 70.0, side='right')
y0 = len(lat_arr) - y0
x0 = np.searchsorted(lon_arr, -70.0, side='left')
zarr_vals = store["band_data"][y0:y0+5, x0:x0+5]
print("Zarr top-left 5x5 values:", zarr_vals)

print("Ratio (GeoTIF / zarr):", geotif_vals / zarr_vals)

GeoTIF transform repr: Affine(0.00025, 0.0, -70.0,
       0.0, -0.00025, 70.0)
GeoTIF top-left 5x5 values: [[266.26367 266.26367 266.26367 266.26367 266.26367]
 [266.26685 266.26685 266.26685 266.26685 266.26685]
 [266.27005 266.27005 266.27005 266.27005 266.27005]
 [266.27322 266.27322 266.27322 266.27322 266.27322]
 [266.2764  266.2764  266.2764  266.2764  266.2764 ]]
Zarr top-left 5x5 values: [[388.55035 388.55035 388.55035 388.55035 388.55035]
 [388.54745 388.54745 388.54745 388.54745 388.54745]
 [388.54453 388.54453 388.54453 388.54453 388.54453]
 [388.54163 388.54163 388.54163 388.54163 388.54163]
 [388.5387  388.5387  388.5387  388.5387  388.5387 ]]
Ratio (GeoTIF / zarr): [[0.68527454 0.68527454 0.68527454 0.68527454 0.68527454]
 [0.68528783 0.68528783 0.68528783 0.68528783 0.68528783]
 [0.68530124 0.68530124 0.68530124 0.68530124 0.68530124]
 [0.68531454 0.68531454 0.68531454 0.68531454 0.68531454]
 [0.6853279  0.6853279  0.6853279  0.6853279  0.6853279 ]]


In [90]:
import rasterio, zarr, fsspec, numpy as np
from src.utilities import constants_and_names as cn

with rasterio.open("s3://gfw2-data/analyses/area_28m/hanson_2013_area_40N_120E.tif") as src:
    print("GeoTIF transform repr:", repr(src.transform))
    geotif_vals = src.read(1, window=rasterio.windows.Window(0, 0, 5, 5))
    print("GeoTIF top-left 5x5 values:", geotif_vals)

fs = fsspec.filesystem("s3", anon=False)
store = zarr.open_group(fs.get_mapper(cn.pixel_area_zarr_path), mode="r")
lat_arr = store["y"][:]
lon_arr = store["x"][:]
y0 = np.searchsorted(lat_arr[::-1], 40.0, side='right')
y0 = len(lat_arr) - y0
x0 = np.searchsorted(lon_arr, 120.0, side='left')
zarr_vals = store["band_data"][y0:y0+5, x0:x0+5]
print("Zarr top-left 5x5 values:", zarr_vals)

print("Ratio (GeoTIF / zarr):", geotif_vals / zarr_vals)

GeoTIF transform repr: Affine(0.00025, 0.0, 120.0,
       0.0, -0.00025, 40.0)
GeoTIF top-left 5x5 values: [[592.6069  592.6069  592.6069  592.6069  592.6069 ]
 [592.609   592.609   592.609   592.609   592.609  ]
 [592.61115 592.61115 592.61115 592.61115 592.61115]
 [592.6133  592.6133  592.6133  592.6133  592.6133 ]
 [592.6154  592.6154  592.6154  592.6154  592.6154 ]]
Zarr top-left 5x5 values: [[592.6069  592.6069  592.6069  592.6069  592.6069 ]
 [592.609   592.609   592.609   592.609   592.609  ]
 [592.61115 592.61115 592.61115 592.61115 592.61115]
 [592.6133  592.6133  592.6133  592.6133  592.6133 ]
 [592.6154  592.6154  592.6154  592.6154  592.6154 ]]
Ratio (GeoTIF / zarr): [[1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]]


In [83]:
import numpy as np

# Expected area for pixels in the top-left 5x5 of 70N_070W (near 70°N)
# Pixel centers start at 70N - 0.5*pixel_size, going south
pixel_size = 0.00025  # degrees
R_earth = 6371007.0  # meters (WGS84 mean radius, or use semi-major axis)

for row in range(5):
    lat = 70.0 - (row + 0.5) * pixel_size  # pixel center latitude
    area = (pixel_size * np.pi / 180) ** 2 * R_earth**2 * np.cos(np.radians(lat))
    print(f"Row {row}, lat {lat:.5f}°N: expected area = {area:.4f} m²")

Row 0, lat 69.99988°N: expected area = 264.3049 m²
Row 1, lat 69.99962°N: expected area = 264.3081 m²
Row 2, lat 69.99938°N: expected area = 264.3112 m²
Row 3, lat 69.99913°N: expected area = 264.3144 m²
Row 4, lat 69.99887°N: expected area = 264.3176 m²


In [92]:
import rasterio
for tile in ["70N_070W", "50N_080W", "40N_130E", "60N_070W"]:
    uri = f"s3://gfw2-data/analyses/area_28m/hanson_2013_area_{tile}.tif"
    with rasterio.open(uri) as ds:
        print(f"{tile}: y_step={ds.transform.e:.8f}, x_step={ds.transform.a:.8f}")

70N_070W: y_step=-0.00025000, x_step=0.00025000
50N_080W: y_step=-0.00025000, x_step=0.00025000
40N_130E: y_step=-0.00025000, x_step=0.00025000
60N_070W: y_step=-0.00025000, x_step=0.00025000


In [1]:
import xarray as xr

for tile in ["70N_070W", "60N_070W"]:
    uri = f"s3://gfw2-data/analyses/area_28m/hanson_2013_area_{tile}.tif"
    ds = xr.open_dataset(uri, engine='rasterio', chunks={'x': 400, 'y': 400})
    y_vals = ds.y.values  # just the coordinate array, tiny
    # read only top-left 3x3 pixels
    data_corner = ds['band_data'].isel(band=0, y=slice(0,3), x=slice(0,3)).values
    print(f"\n{tile}:")
    print(f"  y[0:3]  = {y_vals[:3]}")
    print(f"  y[-3:]  = {y_vals[-3:]}")
    print(f"  data top-left 3x3 = {data_corner}")
    ds.close()

/tmp/ipykernel_23551/459763043.py:5: UserWarning: The specified chunks separate the stored chunks along dimension "x" starting at index 400. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(uri, engine='rasterio', chunks={'x': 400, 'y': 400})



70N_070W:
  y[0:3]  = [69.999875 69.999625 69.999375]
  y[-3:]  = [60.000625 60.000375 60.000125]
  data top-left 3x3 = [[266.26367 266.26367 266.26367]
 [266.26685 266.26685 266.26685]
 [266.27005 266.27005 266.27005]]

60N_070W:
  y[0:3]  = [59.999875 59.999625 59.999375]
  y[-3:]  = [50.000625 50.000375 50.000125]
  data top-left 3x3 = [[388.55328 388.55328 388.55328]
 [388.55618 388.55618 388.55618]
 [388.5591  388.5591  388.5591 ]]


/tmp/ipykernel_23551/459763043.py:5: UserWarning: The specified chunks separate the stored chunks along dimension "x" starting at index 400. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(uri, engine='rasterio', chunks={'x': 400, 'y': 400})


In [43]:
%%time

# To create a wide-format table (with fluxes only, not areas of flux densities).
# Drops a few contextual columns to reduce the number of rows
# Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/699f45bf-932c-8327-95dc-12ded8c246f5

id_cols = [
    # "adm0",
    # "country_name",
    "region_L1",
    # "land_state_node",
    # "land_state_meaning",
    # "land_state_broad_class",
    # "land_state_detailed_class",
    "WDPA",
    "WDPA_type",
    "cont_eco",
    "continent",
    "continent_ecozone",
    "Landmark",
    "starting_composite_primary_forest",
    "year",
    # "tile_id",
    # "area_ha",
    # "density__Mg_ha",
]

df_wide = (
    df.groupby(id_cols + ["analysis_layer"], dropna=False)["value"]
      .sum()
      .unstack("analysis_layer")
      .reset_index()
)

df_wide

CPU times: user 2.74 s, sys: 438 ms, total: 3.18 s
Wall time: 3.17 s


analysis_layer,region_L1,WDPA,WDPA_type,cont_eco,continent,continent_ecozone,Landmark,starting_composite_primary_forest,year,SOC_density__full_extent__0-30cm_MgC_ha,SOC_density__mineral_soil_extent__0-30cm_MgC_ha,SOC_gain__full_extent__0-30cm_MgCO2,SOC_gain__mineral_soil_extent__0-30cm_MgCO2,SOC_loss__full_extent__0-30cm_MgCO2,SOC_loss__mineral_soil_extent__0-30cm_MgCO2,SOC_net__full_extent__0-30cm_MgCO2,SOC_net__mineral_soil_extent__0-30cm_MgCO2
0,Africa,0,NA,0,Unassigned,Unassigned,0,0,2005,3.144888e+06,3.144888e+06,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa,0,NA,0,Unassigned,Unassigned,0,0,2010,3.248894e+06,3.248894e+06,-112562.242188,-112562.242188,36291.031250,36291.031250,-76271.210938,-76271.210938
2,Africa,0,NA,0,Unassigned,Unassigned,0,0,2015,3.224346e+06,3.224346e+06,-63064.789062,-63064.789062,81066.625000,81066.625000,18001.841797,18001.841797
3,Africa,0,NA,0,Unassigned,Unassigned,0,0,2020,3.087219e+06,3.087219e+06,-33849.925781,-33849.925781,134409.953125,134409.953125,100560.015625,100560.015625
4,Africa,0,NA,0,Unassigned,Unassigned,0,0,2022,3.080666e+06,3.080666e+06,-156442.281250,-156442.281250,168456.140625,168456.140625,12013.874023,12013.874023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22570,Unassigned,13,Not Assigned,7014,Europe,Europe-Temperate oceanic forest,1,0,2005,3.269164e+00,1.397565e+00,NaN,NaN,NaN,NaN,NaN,NaN
22571,Unassigned,13,Not Assigned,7014,Europe,Europe-Temperate oceanic forest,1,0,2010,3.265151e+00,1.516507e+00,-0.087224,-0.087224,0.090166,NaN,0.002942,-0.087224
22572,Unassigned,13,Not Assigned,7014,Europe,Europe-Temperate oceanic forest,1,0,2015,3.070279e+00,1.471904e+00,-0.010019,NaN,0.152925,0.032709,0.142907,0.032709
22573,Unassigned,13,Not Assigned,7014,Europe,Europe-Temperate oceanic forest,1,0,2020,2.613391e+00,8.920630e-01,-0.090166,NaN,0.425217,0.425217,0.335051,0.425217


In [44]:
%%time
# Number of options for each contextual column

summary = {
    col: df[col].nunique()
    for col in df.columns
    if col not in ["index", "value", "analysis_layer", "area_ha", "density__Mg_ha"]
}

pd.Series(summary).sort_values(ascending=False)

CPU times: user 2.66 s, sys: 131 ms, total: 2.79 s
Wall time: 2.8 s


tile_id                              275
watershed                            225
watershed_name                       225
country_name                         217
adm0                                 217
cont_eco                              76
continent_ecozone                     76
ecozone                               24
region_L2_L3                          23
WDPA                                  13
WDPA_type                             13
continent                              9
drivers_of_TCL_1_km                    8
driver_1km_text                        8
region_L1                              8
year                                   5
climate_domain                         4
WDPA_high_protection                   3
KBA                                    2
starting_composite_primary_forest      2
Landmark                               2
gas                                    1
dtype: int64

In [46]:
# Keeping this in SOC notebook for parity with vegetation notebook
df_outputs_dropped = df.copy()

In [48]:
%%time
# Drops contextual layers sequentially to get a sense of how many combination rows are added by including each one,
# i.e. how many rows are lost when I drop that one column, given that all the other columns are still present.
# per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/699f45bf-932c-8327-95dc-12ded8c246f5

# Contextual layers that are essentially text versions of others, so they need to be dropped in order to assess how much complexity each contextual layer gives 
# (since they are redundant in terms of complexity)
df_by_context = df_outputs_dropped.drop(columns=['watershed_name', 
                                                 'country_name', 'region_L1', 'region_L2_L3', 
                                                 'WDPA_type', 'WDPA_high_protection',
                                                 'continent_ecozone', 'continent', 'ecozone', 'climate_domain',
                                                 'driver_1km_text'])

base = len(df_by_context)
print(f"Rows with all contextual layers: {base}")

cols_to_check = ['tile_id', 'watershed', 'adm0', 'cont_eco', 'WDPA', 'drivers_of_TCL_1_km', 'year', 'Landmark', 'starting_composite_primary_forest']

for col in cols_to_check:

    # Drops specified contextual layers to reduce the number of rows in the table
    cols_to_sum = ["value", "area_ha", "density__Mg_ha"]
    
    group_cols = [
        c for c in df_by_context.columns
        if c not in [col] + cols_to_sum  # Contextual columns to drop
    ]
    
    df_agg = (
        df_by_context.groupby(group_cols, dropna=False)
          .size()
          .reset_index()
    )
    reduced=len(df_agg)

    print(f"{col:35s} removing it reduces rows by {base - reduced}")

Rows with all contextual layers: 7987511
tile_id                             removing it reduces rows by 2454868
watershed                           removing it reduces rows by 2877482
adm0                                removing it reduces rows by 2311265
cont_eco                            removing it reduces rows by 2831642
WDPA                                removing it reduces rows by 4664337
drivers_of_TCL_1_km                 removing it reduces rows by 5502118
year                                removing it reduces rows by 6075969
Landmark                            removing it reduces rows by 1261951
starting_composite_primary_forest   removing it reduces rows by 2703675
CPU times: user 29.8 s, sys: 6.11 s, total: 35.9 s
Wall time: 36.1 s


In [49]:
# Number of rows for each analysis layer 
df_outputs_dropped["analysis_layer"].value_counts()

analysis_layer
SOC_density__full_extent__0-30cm_MgC_ha            1214295
SOC_density__mineral_soil_extent__0-30cm_MgC_ha    1178025
SOC_net__full_extent__0-30cm_MgCO2                  970024
SOC_loss__full_extent__0-30cm_MgCO2                 941388
SOC_net__mineral_soil_extent__0-30cm_MgCO2          940948
SOC_gain__full_extent__0-30cm_MgCO2                 930681
SOC_loss__mineral_soil_extent__0-30cm_MgCO2         912045
SOC_gain__mineral_soil_extent__0-30cm_MgCO2         900105
Name: count, dtype: int64

In [78]:
# Drops various contextual layers to reduce df size for PyGWalker
print(f"Columns before dropping: {df_outputs_dropped.columns}")
# Comment out the contextual layers to keep. 
# Layers on the same line are redundant with each other; they need to be dropped or retained together for full effect.
contextual_layers_to_drop = [   
                             # 'tile_id', 
                             cn.adm0_pattern, 'country_name', 
                             cn.WDPA_pattern, 'WDPA_type', 
                             'WDPA_high_protection',
                             cn.cont_eco_zstats_pattern, 'continent_ecozone', 'continent', 
                             cn.landmark_pattern, 
                             cn.starting_composite_primary_forest_pattern,
                             # cn.land_state_pattern, 'land_state_meaning', 
                             # 'land_state_broad_class', 
                             # 'land_state_detailed_class',
                             'watershed', 'watershed_name',
                             'region_L1', 'region_L2_L3'
                            ]
df_outputs_context_dropped = df_outputs_dropped.drop(columns=contextual_layers_to_drop)

base = len(df_outputs_context_dropped)
print(f"Rows in df with outputs removed, with all contextual layers: {base}")

# Drops specified contextual layers to reduce the number of rows in the table (although dropping adm0 or land_state seems to remove only a few 10s of thousands of rows)
cols_to_sum = ["value", "area_ha", "density__Mg_ha"]

group_cols = [
    c for c in df_outputs_context_dropped.columns
    if c not in contextual_layers_to_drop + cols_to_sum  # Contextual columns to drop
]
# print("group_cols:", group_cols)

df_outputs_context_dropped_agg = (
    df_outputs_context_dropped.groupby(group_cols, dropna=False)
      .sum(numeric_only=True)
      .reset_index()
)
reduced=len(df_outputs_context_dropped_agg)

print(f"Columns after dropping: {df_outputs_context_dropped_agg.columns}")
print(f"Removing tile_id reduces rows by {base - reduced}")
print(f"Rows in output with contextual layers dropped: {reduced}")

Columns before dropping: Index(['analysis_layer', 'adm0', 'WDPA', 'cont_eco', 'Landmark',
       'starting_composite_primary_forest', 'KBA', 'watershed',
       'drivers_of_TCL_1_km', 'year', 'value', 'tile_id', 'area_ha', 'gas',
       'country_name', 'region_L1', 'region_L2_L3', 'continent', 'ecozone',
       'continent_ecozone', 'climate_domain', 'watershed_name', 'WDPA_type',
       'WDPA_high_protection', 'driver_1km_text', 'density__Mg_ha'],
      dtype='object')
Rows in df with outputs removed, with all contextual layers: 7987511
Columns after dropping: Index(['analysis_layer', 'KBA', 'drivers_of_TCL_1_km', 'year', 'tile_id',
       'gas', 'ecozone', 'climate_domain', 'driver_1km_text', 'value',
       'area_ha', 'density__Mg_ha'],
      dtype='object')
Removing tile_id reduces rows by 7608825
Rows in output with contextual layers dropped: 378686


In [79]:
%%time

# Test sums for original and simplified tables. Values should be identical or very close.
year = 2022
variable = 'SOC_net__mineral_soil_extent__0-30cm_MgCO2'  
print(f"Full df:        {int(df[(df['year'] == year) & (df['analysis_layer'] ==variable)]['value'].sum())}")
print(f"Simplified df:  {int(df_outputs_context_dropped_agg[(df_outputs_context_dropped_agg['year'] == year) & (df_outputs_context_dropped_agg['analysis_layer']==variable)]['value'].sum())}")
chunk_stat_sum = 2252877819 # Chunk stats value from Excel pivot table
print(f"Difference between zonal stats and chunk stats sums: {int(df[(df['year'] == year) & (df['analysis_layer'] ==variable)]['value'].sum() - chunk_stat_sum)}")

year = 2020
variable = 'SOC_net__mineral_soil_extent__0-30cm_MgCO2'  
print(f"Full df:        {int(df[(df['year'] == year) & (df['analysis_layer'] ==variable)]['value'].sum())}")
print(f"Simplified df:  {int(df_outputs_context_dropped_agg[(df_outputs_context_dropped_agg['year'] == year) & (df_outputs_context_dropped_agg['analysis_layer']==variable)]['value'].sum())}")
chunk_stat_sum = 1264590854 # Chunk stats value from Excel pivot table
print(f"Difference between zonal stats and chunk stats sums: {int(df[(df['year'] == year) & (df['analysis_layer'] ==variable)]['value'].sum() - chunk_stat_sum)}")

year = 2020
variable = 'SOC_loss__mineral_soil_extent__0-30cm_MgCO2'  
print(f"Full df:        {int(df[(df['year'] == year) & (df['analysis_layer'] ==variable)]['value'].sum())}")
print(f"Simplified df:  {int(df_outputs_context_dropped_agg[(df_outputs_context_dropped_agg['year'] == year) & (df_outputs_context_dropped_agg['analysis_layer']==variable)]['value'].sum())}")
chunk_stat_sum = 6760376953 # Chunk stats value from Excel pivot table
print(f"Difference between zonal stats and chunk stats sums: {int(df[(df['year'] == year) & (df['analysis_layer'] ==variable)]['value'].sum() - chunk_stat_sum)}")

year = 2015
variable = 'SOC_gain__mineral_soil_extent__0-30cm_MgCO2'  
print(f"Full df:       {int(df[(df['year'] == year) & (df['analysis_layer'] ==variable)]['value'].sum())}")
print(f"Simplified df: {int(df_outputs_context_dropped_agg[(df_outputs_context_dropped_agg['year'] == year) & (df_outputs_context_dropped_agg['analysis_layer']==variable)]['value'].sum())}")
chunk_stat_sum = -4796027564 # Chunk stats value from Excel pivot table
print(f"Difference between zonal stats and chunk stats sums: {int(df[(df['year'] == year) & (df['analysis_layer'] ==variable)]['value'].sum() - chunk_stat_sum)}")

Full df:        2248729344
Simplified df:  2248729600
Difference between zonal stats and chunk stats sums: -4148480
Full df:        1263067392
Simplified df:  1263067520
Difference between zonal stats and chunk stats sums: -1523456
Full df:        6761364480
Simplified df:  6761363968
Difference between zonal stats and chunk stats sums: 987648
Full df:       -4796726272
Simplified df: -4796726784
Difference between zonal stats and chunk stats sums: -698880
CPU times: user 2.13 s, sys: 8.32 ms, total: 2.14 s
Wall time: 2.15 s


In [80]:
# Loads in PyGWalker 
walker = pyg.walk(df_outputs_context_dropped_agg)

Box(children=(HTML(value='<div id="ifr-pyg-1" style="height: auto">\n    <head>\n        <meta http-equiv="Con…